In [1]:
import os
import yaml
from ultralytics import YOLO, settings
import torch

settings.update({'datasets_dir': r'E:\Projects\CCTV_horkang'})
print("Settings updated to project directory.")

Settings updated to project directory.


In [2]:
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    # Verify it sees the 5060 and the 12.0 capability
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
    
    # Critical test: perform a calculation on the GPU
    # If this doesn't crash, your 5060 is ready!
    test_tensor = torch.randn(1024, 1024).cuda()
    result = test_tensor @ test_tensor
    print("✅ GPU calculation successful!")

CUDA available: True
GPU: NVIDIA GeForce RTX 5060
Compute Capability: (12, 0)
✅ GPU calculation successful!


In [5]:
device = '0' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if device=='0' else 'CPU'})")

Using device: 0 (NVIDIA GeForce RTX 5060)


In [12]:
project_root = r'E:\Projects\CCTV_horkang\train_dataset'
train_images = os.path.join(project_root, 'train', 'images')
val_images = os.path.join(project_root, 'valid', 'images')
test_images = os.path.join(project_root, 'test', 'images')

data_config = {
    'train': train_images,
    'val': val_images,
    'test': test_images,
    'nc': 1,
    'names': ['human']
}

yaml_path = os.path.join(project_root, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)


In [9]:
model = YOLO('yolov8n.pt')
model.train(
    data=r'E:\Projects\CCTV_horkang\train_dataset\data.yaml',
    epochs=100,      # High epochs for better accuracy
    imgsz=640,       # Standard resolution
    batch=16,        # Adjust to 8 or 4 if you get "Out of Memory"
    device=device,   # Uses your NVIDIA GPU
    workers=4,       # Number of CPU threads loading data
    name='human_detector_v1',
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.24 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.10.19 torch-2.12.0.dev20260320+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\Projects\CCTV_horkang\train_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000024C3040B7F0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [15]:
best_model = YOLO('runs/detect/human_detector_v1/weights/best.pt')
metrics = best_model.val(
    data=r'E:\Projects\CCTV_horkang\train_dataset\data.yaml', 
    split='test',
    workers=0,
    device=0
)
print("--- Test Set Results ---")
print(f"mAP50-95: {metrics.box.map:.3f}") # Overall accuracy
print(f"mAP50:    {metrics.box.map50:.3f}") # Accuracy at 50% overlap (Standard)
print(f"Precision: {metrics.box.mp:.3f}")   # How many detections were correct
print(f"Recall:    {metrics.box.mr:.3f}")

Ultralytics 8.4.14  Python-3.10.19 torch-2.12.0.dev20260320+cu128 CUDA:0 (NVIDIA GeForce RTX 5060, 8151MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 112.447.6 MB/s, size: 29.7 KB)
val: Scanning E:\Projects\CCTV_horkang\train_dataset\test\labels.cache... 1078 images, 3 backgrounds, 16 corrupt: 100% ━━━━━━━━━━━━ 1078/1078  0.0s
val: E:\Projects\CCTV_horkang\train_dataset\test\images\e12d866ca225bf47c35481ef4947822c575cb668_jpg.rf.a1564ab7c20f28db2e7abe101e90d705.jpg: ignoring corrupt image/label: non-normalized or out of bounds coordinates [1.0868497]
val: E:\Projects\CCTV_horkang\train_dataset\test\images\train_all_data--105-_jpg.rf.ef0aa61a8c8618030c4ae958a50d37cc.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: E:\Projects\CCTV_horkang\train_dataset\test\images\train_all_data--108-_jpg.rf.e585280f12665fe7b15b839a6db8a337.jpg: ignoring

Exception in thread Thread-84 (plot_images):
Traceback (most recent call last):
  File "c:\Users\Ausak\anaconda3\envs\cctv_env\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Ausak\anaconda3\envs\cctv_env\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Ausak\anaconda3\envs\cctv_env\lib\site-packages\ultralytics\utils\plotting.py", line 761, in plot_images
    mosaic = cv2.resize(mosaic, tuple(int(x * ns) for x in (w, h)))
cv2.error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\core\src\alloc.cpp:73: error: (-4:Insufficient memory) Failed to allocate 11059200 bytes in function 'cv::OutOfMemoryError'



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 6.9it/s 9.7s0.1s
WARNING ConfusionMatrix plot failure: Unable to allocate 18.5 MiB for an array with shape (1557, 1557) and data type int64
                   all       1062       3453      0.734      0.583      0.656      0.397
Speed: 0.2ms preprocess, 1.9ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to E:\Projects\CCTV_horkang\runs\detect\val3
--- Test Set Results ---
mAP50-95: 0.397
mAP50:    0.656
Precision: 0.734
Recall:    0.583
